# Mamba3 SISO 40M — English smoke

SISO control architecture. The dataset profile is shared with GQA/MIMO, while the official Mamba3 SISO gate and configuration remain separate.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
REPO=Path('/workspace/murmur-science')
if not REPO.exists(): subprocess.run(['git','clone','--branch','codex/mamba3-siso-smoke','https://github.com/orkrs/murmur-science.git',str(REPO)],check=True)
%cd /workspace/murmur-science
sys.path.insert(0,str(Path.cwd()/'src'))
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
%pip install -q datasets sentencepiece pyarrow pandas einops ninja
%pip install -q --upgrade 'tilelang==0.1.9' 'apache-tvm-ffi<=0.1.12' 'quack-kernels>=0.3.4' 'triton>=3.5.0'
import os; os.environ['MAMBA_FORCE_BUILD']='TRUE'
%pip install -q --no-cache-dir --no-deps --force-reinstall --no-build-isolation git+https://github.com/state-spaces/mamba.git@main

In [ ]:
from murmur.config import load_run_config
from murmur.model.mixers.mamba3 import Mamba3Block
config=load_run_config(Path('configs/smoke_mamba3_siso.toml'))
assert config.model.mixer=='mamba3'
gate=Mamba3Block(128,256,64,64,1e-5,1.0,0,is_mimo=False).cuda().half().train()
x=torch.randn(1,512,128,device='cuda',dtype=torch.float16,requires_grad=True)
loss=gate(x)[0].float().square().mean(); loss.backward()
print('Verified official Mamba3 SISO forward/backward gate')

In [ ]:
subprocess.run([sys.executable,'scripts/build_hf_mix.py','--profile','english_smoke','--output','artifacts/english_smoke_corpus','--max-tokens','2400000'],check=True)
!python scripts/train_tokenizer.py --corpus artifacts/english_smoke_corpus/corpus.txt --output artifacts/english_smoke_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/smoke_mamba3_siso.toml --tokenizer artifacts/english_smoke_tokenizer.model --train-input artifacts/english_smoke_corpus/train.jsonl --val-input artifacts/english_smoke_corpus/val.jsonl --output artifacts/english_smoke_data

In [ ]:
run_dir=Path('artifacts/runs/siso_english_smoke')
subprocess.run([sys.executable,'scripts/train.py','--config','configs/smoke_mamba3_siso.toml','--run-dir',str(run_dir),'--device','cuda'],check=True)
assert (run_dir/'checkpoints'/'last'/'COMPLETED').exists()
print('SISO English smoke checkpoint ready')